# 07. Testing Strategy — unit, integration, eval을 분리하기

## 학습 목표

- Agent 테스트를 **unit**, **integration**, **eval**로 분리합니다.
- 외부 API 없이 deterministic unit test를 작성합니다.
- Agent Evals와 smoke test가 맡는 역할을 구분합니다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# LangSmith / Langfuse 설정 — 키가 없으면 비활성 상태로 둡니다.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import HumanMessage

model = FakeListChatModel(responses=["테스트 응답"])
model.invoke([HumanMessage(content="ping")]).content

## 7.1 테스트 피라미드

| 층 | 목적 | 외부 서비스 |
|---|---|---|
| unit | 함수, prompt, parser, graph node | 없음 |
| integration | provider, DB, retriever, server 연결 | opt-in |
| eval | trajectory/품질/회귀 평가 | 선택적 LLM judge |

## 7.2 Unit test: 순수 함수부터 고정

LLM 호출 전에 입력 정규화, tool routing, state update 같은 작은 단위를 먼저 고정합니다.

In [ ]:
def normalize_question(text: str) -> str:
    return " ".join(text.strip().lower().split())

assert normalize_question("  Hello   Agent  ") == "hello agent"
print("unit test passed")

## 7.3 Integration gate

외부 서비스 테스트는 키가 있을 때만 실행합니다. 기본 smoke에서 실패하면 안 됩니다.

In [ ]:
def can_run_openai_integration() -> bool:
    return bool(os.getenv("OPENAI_API_KEY")) and os.getenv("RUN_LIVE_TESTS") == "1"

print("run live integration:", can_run_openai_integration())

## 7.4 Eval은 “정답 문자열”보다 경로를 본다

Agent는 같은 답을 여러 문장으로 말할 수 있으므로, tool trajectory와 rubric을 함께 봅니다.

In [ ]:
trajectory = [
    {"role": "user", "content": "서울 날씨"},
    {"role": "assistant", "tool_calls": [{"name": "get_weather"}]},
    {"role": "tool", "name": "get_weather", "content": "맑음"},
]

assert trajectory[1]["tool_calls"][0]["name"] == "get_weather"

## 7.5 CI/수동 검증 분리

| 검증 | 기본 CI | 수동 opt-in |
|---|---|---|
| notebook JSON/cell ID | yes | yes |
| deterministic unit cells | yes | yes |
| provider integration | no | `RUN_LIVE_TESTS=1` |
| LangSmith mutation | no | `--allow-langsmith-mutations` |

---

## 정리

| 항목 | 내용 |
|---|---|
| **다룬 기술** | fake chat model, unit assertion, integration gate, trajectory check |
| **핵심 개념** | 테스트 실패 원인을 작게 만들려면 unit/integration/eval을 섞지 않아야 합니다. |
| **다음 단계** | `06_agent_evals.ipynb`, `08_langgraph_testing.ipynb` 후보 |

**참고 문서:**
- `docs/langchain/test/index.md`
- `docs/langchain/test/unit-testing.md`
- `docs/langchain/test/integration-testing.md`
- `docs/langchain/test/evals.md`
- `docs/langgraph/test.md`